# T15-国债收益率走势阶段分析

## 任务概述

国债收益率走势阶段分析工具，用于分析10年期国债收益率的历史走势，支持交互式标注和周期划分。

**主要功能**:
- 10年期国债收益率数据获取
- 移动平均线计算（20日、60日）
- 交互式周期标注
- 收益率变化统计
- 数据导出

---

## 1. 环境准备与依赖导入

In [ ]:
# 标准库导入
import os
import sys
from datetime import datetime

# 数据处理
import pandas as pd
import numpy as np
import sqlalchemy
from sqlalchemy import pool

# 可视化
import matplotlib.pyplot as plt
from matplotlib.widgets import Button
from matplotlib.dates import DateFormatter, YearLocator, date2num, num2date

# 配置中文字体
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

# 导入配置
from config import (
    DATABASE_URL,
    START_DATE,
    END_DATE,
    MA_SHORT_WINDOW,
    MA_LONG_WINDOW,
    CHART_CONFIG,
    COLORS,
    OUTPUT_FILENAME,
    get_yield_query
)

print("环境准备完成!")
print(f"分析日期范围: {START_DATE} 至 {END_DATE}")

## 2. 数据获取与加载

In [ ]:
def load_yield_data(start_date: str, end_date: str) -> pd.DataFrame:
    """
    从数据库加载10年期国债收益率数据
    
    Args:
        start_date: 开始日期
        end_date: 结束日期
    
    Returns:
        包含日期和收益率列的DataFrame
    """
    # 创建数据库引擎
    engine = sqlalchemy.create_engine(
        DATABASE_URL,
        poolclass=sqlalchemy.pool.NullPool
    )
    
    # 获取查询SQL
    query = get_yield_query(start_date, end_date)
    
    # 执行查询
    data = pd.read_sql(query, engine)
    data['dt'] = pd.to_datetime(data['dt'])
    
    # 关闭连接
    engine.dispose()
    
    print(f"数据加载完成，共 {len(data)} 条记录")
    print(f"日期范围: {data['dt'].min().strftime('%Y-%m-%d')} 至 {data['dt'].max().strftime('%Y-%m-%d')}")
    
    return data

# 加载数据
data = load_yield_data(START_DATE, END_DATE)

# 显示数据预览
data.head(10)

## 3. 数据处理 - 移动平均线计算

In [ ]:
def calculate_moving_averages(df: pd.DataFrame, short_window: int, long_window: int) -> pd.DataFrame:
    """
    计算移动平均线
    
    Args:
        df: 原始数据DataFrame
        short_window: 短期均线窗口
        long_window: 长期均线窗口
    
    Returns:
        添加了移动平均线列的DataFrame
    """
    df = df.copy()
    df[f'MA{short_window}'] = df['收益率'].rolling(window=short_window).mean()
    df[f'MA{long_window}'] = df['收益率'].rolling(window=long_window).mean()
    
    return df

# 计算移动平均线
data = calculate_moving_averages(data, MA_SHORT_WINDOW, MA_LONG_WINDOW)

# 显示处理后的数据
data.tail(10)

In [ ]:
# 数据统计摘要
summary = {
    '收益率均值': data['收益率'].mean(),
    '收益率标准差': data['收益率'].std(),
    '收益率最大值': data['收益率'].max(),
    '收益率最小值': data['收益率'].min(),
    '收益率中位数': data['收益率'].median(),
}

print("="*50)
print("数据统计摘要")
print("="*50)
for key, value in summary.items():
    print(f"{key}: {value:.4f}%")

## 4. 可视化 - 收益率走势图

In [ ]:
def plot_yield_curve(df: pd.DataFrame):
    """
    绘制收益率走势图（静态版本）
    
    Args:
        df: 包含收益率和移动平均线的数据
    """
    fig, ax = plt.subplots(figsize=CHART_CONFIG['figsize'])
    
    # 绘制收益率曲线
    ax.plot(
        df['dt'], 
        df['收益率'], 
        linewidth=CHART_CONFIG['line_width_raw'],
        label='收益率', 
        color=COLORS['yield_line'], 
        alpha=CHART_CONFIG['alpha_raw']
    )
    
    # 绘制移动平均线
    ax.plot(
        df['dt'], 
        df[f'MA{MA_SHORT_WINDOW}'], 
        linewidth=CHART_CONFIG['line_width_ma'],
        label=f'{MA_SHORT_WINDOW}日均线', 
        color=COLORS['ma20']
    )
    
    ax.plot(
        df['dt'], 
        df[f'MA{MA_LONG_WINDOW}'], 
        linewidth=CHART_CONFIG['line_width_ma'],
        label=f'{MA_LONG_WINDOW}日均线', 
        color=COLORS['ma60']
    )
    
    # 设置标题和标签
    ax.set_title(CHART_CONFIG['title'], pad=20, fontsize=12)
    ax.set_xlabel(CHART_CONFIG['xlabel'], fontsize=10)
    ax.set_ylabel(CHART_CONFIG['ylabel'], fontsize=10)
    ax.grid(True, linestyle='--', alpha=CHART_CONFIG['grid_alpha'])
    
    # 设置x轴时间格式
    ax.xaxis.set_major_locator(YearLocator())
    ax.xaxis.set_major_formatter(DateFormatter('%Y'))
    plt.xticks(rotation=45)
    
    # 添加图例
    ax.legend(loc='center left', bbox_to_anchor=(1, 0.5))
    
    plt.tight_layout()
    plt.show()

# 绘制静态图表
plot_yield_curve(data)

## 5. 交互式周期标注功能

In [ ]:
class YieldCurveAnnotator:
    """
    收益率曲线交互式标注器
    
    用于在图表上交互式标注周期起点和终点
    """
    
    def __init__(self, df: pd.DataFrame):
        """
        初始化标注器
        
        Args:
            df: 包含收益率数据的DataFrame
        """
        self.data = df
        self.points = []  # 存储标注点 (日期, 收益率, 索引)
        self.annotations = []  # 存储标注文本
        self.markers = []  # 存储标记点对象
        self.current_cycle = 1
        self.fig = None
        self.ax = None
        
    def setup_plot(self):
        """设置图形界面"""
        self.fig, self.ax = plt.subplots(figsize=CHART_CONFIG['figsize'])
        plt.subplots_adjust(bottom=0.2, right=0.85)
        
        # 绘制收益率曲线
        self.ax.plot(
            self.data['dt'], 
            self.data['收益率'], 
            linewidth=CHART_CONFIG['line_width_raw'],
            label='收益率', 
            color=COLORS['yield_line'], 
            alpha=CHART_CONFIG['alpha_raw']
        )
        
        # 绘制移动平均线
        self.ax.plot(
            self.data['dt'], 
            self.data[f'MA{MA_SHORT_WINDOW}'], 
            linewidth=CHART_CONFIG['line_width_ma'],
            label=f'{MA_SHORT_WINDOW}日均线', 
            color=COLORS['ma20']
        )
        
        self.ax.plot(
            self.data['dt'], 
            self.data[f'MA{MA_LONG_WINDOW}'], 
            linewidth=CHART_CONFIG['line_width_ma'],
            label=f'{MA_LONG_WINDOW}日均线', 
            color=COLORS['ma60']
        )
        
        # 设置标题和标签
        self.ax.set_title(CHART_CONFIG['title'], pad=20, fontsize=12)
        self.ax.set_xlabel(CHART_CONFIG['xlabel'], fontsize=10)
        self.ax.set_ylabel(CHART_CONFIG['ylabel'], fontsize=10)
        self.ax.grid(True, linestyle='--', alpha=CHART_CONFIG['grid_alpha'])
        
        # 设置x轴时间格式
        self.ax.xaxis.set_major_locator(YearLocator())
        self.ax.xaxis.set_major_formatter(DateFormatter('%Y'))
        plt.xticks(rotation=45)
        
        # 添加图例
        self.ax.legend(loc='center left', bbox_to_anchor=(1, 0.5))
        
        # 添加按钮
        ax_clear = plt.axes([0.7, 0.05, 0.1, 0.075])
        ax_save = plt.axes([0.81, 0.05, 0.1, 0.075])
        
        btn_clear = Button(ax_clear, '清除标注')
        btn_save = Button(ax_save, '保存标注')
        
        # 绑定事件
        self.fig.canvas.mpl_connect('button_press_event', self._onclick)
        btn_clear.on_clicked(self._clear_points)
        btn_save.on_clicked(self._save_points)
        
        return self.fig, self.ax
    
    def _onclick(self, event):
        """点击事件处理"""
        if event.inaxes != self.ax:
            return
        
        # 获取最接近点击位置的数据点
        x_date = num2date(event.xdata).replace(tzinfo=None)
        idx = abs(self.data['dt'] - x_date).argmin()
        x = self.data['dt'].iloc[idx]
        y = self.data['收益率'].iloc[idx]
        
        # 添加标注点
        marker = self.ax.plot(
            x, y, 'o', 
            markersize=10, 
            markerfacecolor=COLORS['marker'], 
            markeredgecolor=COLORS['marker_edge'], 
            markeredgewidth=2
        )[0]
        self.markers.append(marker)
        self.points.append((x, y, idx))
        
        # 添加文本标注
        bbox_props = dict(
            boxstyle="round,pad=0.3", 
            fc=COLORS['annotation_bg'], 
            ec=COLORS['annotation_edge'], 
            alpha=0.8
        )
        
        if len(self.points) % 2 == 1:
            annotation = self.ax.annotate(
                f'周期{self.current_cycle}起点\n{x.strftime("%Y-%m-%d")}\n{y:.2f}%',
                (x, y),
                xytext=(10, 10),
                textcoords='offset points',
                bbox=bbox_props,
                fontsize=9
            )
        else:
            annotation = self.ax.annotate(
                f'周期{self.current_cycle}终点\n{x.strftime("%Y-%m-%d")}\n{y:.2f}%',
                (x, y),
                xytext=(10, -10),
                textcoords='offset points',
                bbox=bbox_props,
                fontsize=9
            )
            self.current_cycle += 1
        
        self.annotations.append(annotation)
        plt.draw()
    
    def _clear_points(self, event):
        """清除所有标注点"""
        for marker in self.markers:
            marker.remove()
        for annotation in self.annotations:
            annotation.remove()
        
        self.markers.clear()
        self.points.clear()
        self.annotations.clear()
        self.current_cycle = 1
        plt.draw()
        print("已清除所有标注")
    
    def _save_points(self, event):
        """保存标注点信息到CSV"""
        marked_points = []
        
        for i in range(0, len(self.points), 2):
            if i + 1 < len(self.points):
                cycle_num = i // 2 + 1
                start_date, start_yield, start_idx = self.points[i]
                end_date, end_yield, end_idx = self.points[i+1]
                
                marked_points.append({
                    'cycle': cycle_num,
                    'start_date': start_date.strftime('%Y-%m-%d'),
                    'end_date': end_date.strftime('%Y-%m-%d'),
                    'start_yield': round(start_yield, 4),
                    'end_yield': round(end_yield, 4),
                    'yield_change': round(end_yield - start_yield, 4)
                })
        
        if marked_points:
            df = pd.DataFrame(marked_points)
            df.to_csv(OUTPUT_FILENAME, index=False, encoding='utf-8-sig')
            print(f"\n标注已保存到: {OUTPUT_FILENAME}")
            print("\n已标注的周期:")
            print(df.to_string(index=False))
        else:
            print("没有完整的周期标注可保存")
    
    def show(self):
        """显示交互式图表"""
        self.setup_plot()
        plt.show()
        return self

print("交互式标注器类定义完成!")

## 6. 启动交互式标注

**使用说明**:
1. 运行下方单元格后显示收益率走势图
2. 在图表上点击标注周期起点
3. 再次点击标注周期终点
4. 重复步骤2-3标注更多周期
5. 点击"保存标注"按钮导出数据到CSV
6. 点击"清除标注"按钮可重新开始

In [ ]:
# 创建并显示交互式标注器
# 注意：需要GUI环境支持
annotator = YieldCurveAnnotator(data)
annotator.show()

## 7. 查看已保存的标注结果

In [ ]:
# 检查是否存在标注结果文件
if os.path.exists(OUTPUT_FILENAME):
    saved_data = pd.read_csv(OUTPUT_FILENAME, encoding='utf-8-sig')
    print(f"已加载标注文件: {OUTPUT_FILENAME}")
    print(f"共 {len(saved_data)} 个周期")
    print("\n标注详情:")
    display(saved_data)
    
    # 统计汇总
    print("\n" + "="*50)
    print("周期统计汇总")
    print("="*50)
    print(f"下行周期数量: {len(saved_data[saved_data['yield_change'] < 0])}")
    print(f"上行周期数量: {len(saved_data[saved_data['yield_change'] > 0])}")
    print(f"平均收益率变化: {saved_data['yield_change'].mean():.4f}%")
else:
    print(f"标注文件 {OUTPUT_FILENAME} 不存在")
    print("请先运行交互式标注功能并保存标注结果")

## 8. 总结

本Notebook完成了以下功能:

1. **数据获取**: 从数据库加载10年期国债收益率数据
2. **数据处理**: 计算20日和60日移动平均线
3. **可视化**: 绘制收益率走势图
4. **交互标注**: 支持点击标注周期起点和终点
5. **数据导出**: 将标注结果保存为CSV文件

**注意事项**:
- 交互式标注功能需要GUI环境支持
- 配置文件使用环境变量，无硬编码密码
- 标注点按顺序成对出现（起点-终点）